# Stand-Side Aircraft Variant Verification

**Capstone Project 6, Agentic AI Systems. Brian Paul Roche.**

This notebook runs the system and shows what it actually does rather than describing it.

Four agents decide whether automated visual docking guidance may be released for an
arriving aircraft. A Stand Coordinator holds no tools at all and can only ask its three
workers, each of which holds exactly one capability. The terminal decision is not taken by
any of them. It is taken by a gate written in Python that reads the tool outputs directly.

The reason for that shape is the thing this notebook ends up demonstrating, so it is worth
stating first: **authority in this system is granted by which tools an agent holds, never
by what its prompt says.** A prompt can be argued with. A tool allocation cannot.

## 1. Configuration

Every policy constant lives in one module, so the operating point of the system is a
stated parameter rather than a number buried in the logic. The confidence floor is the one
that matters, and it is not a number chosen because it looked careful. It comes from the
operating curve I produced in Project 7.

In [1]:
import json, csv, os, sys
sys.path.insert(0, os.getcwd())

from config import CONFIG, VARIANTS, OUT_OF_FAMILY, BRIDGE_ENVELOPE, REASONS

for k in ["confidence_floor", "in_family_prefix", "n_stands", "seed", "model_id",
          "measured_visual_accuracy", "measured_chance_rate"]:
    print(f"{k:28} {CONFIG[k]}")
print()
print("bridge envelope, door 1L metres from the nose")
for v, m in BRIDGE_ENVELOPE.items():
    print(f"   {v}   {m}")

confidence_floor             0.6
in_family_prefix             737
n_stands                     40
seed                         42
model_id                     gpt-4o-mini
measured_visual_accuracy     0.192
measured_chance_rate         0.125

bridge envelope, door 1L metres from the nose
   737-200   9.1
   737-300   10.1
   737-400   11.6
   737-500   9.6
   737-600   11.2
   737-700   12.1
   737-800   13.4
   737-900   14.6


## 2. The tools, and the purpose of each

Three tools exist and each belongs to exactly one worker. This module is where authority
lives: an agent can do what its tools let it do and nothing else.

Each looks its subject up by identifier and returns an explicit error when asked about
something it does not hold, rather than returning a neighbouring answer. That is what stops
an agent which asks about the wrong stand from being handed the right stand's facts by
accident.

In [2]:
from helpers import load_scenarios
from tools import (load_stands, identify_variant_from_image,
                   lookup_expected_variant, bridge_envelope_for)

rows = load_scenarios(CONFIG["scenario_file"])
load_stands(rows)
print(f"{len(rows)} stand scenarios loaded, seed {CONFIG['seed']}\n")

print("the stand camera on S001:")
print("  ", identify_variant_from_image("S001"))
print("\nthe arrival record for S001:")
print("  ", lookup_expected_variant("S001"))
print("\nthe bridge envelope for a 737-800:")
print("  ", bridge_envelope_for("737-800"))
print("\nand a stand that does not exist, which is refused rather than guessed:")
print("  ", identify_variant_from_image("S999"))

40 stand scenarios loaded, seed 42

the stand camera on S001:
   {"stand": "S001", "variant": "737-600", "confidence": 0.47, "note": "this channel measured 0.192 accuracy against a chance rate of 0.125 and cannot authorise a release"}

the arrival record for S001:
   {"stand": "S001", "record_variant": "737-200"}

the bridge envelope for a 737-800:
   {"variant": "737-800", "door_1L_metres_from_nose": 13.4, "on_file": true}

and a stand that does not exist, which is refused rather than guessed:
   {"error": "no camera feed for stand S999", "known": "40 stands, S001 upwards"}


## 3. The gate

The gate is an ordinary Python function. It applies five rules in a fixed order and
returns a verdict with a reason code attached. It may be called by the system. It cannot
be overruled by it, and no prompt reaches it.

The order matters. An airframe outside the in-service family is refused before any
comparison is attempted, because a system that does not recognise the aircraft type has no
business reasoning about its doors. Only if the two channels name the same variant does
the confidence floor come into play at all.

In [3]:
from helpers import gate_verdict

demo = [
    ("737-800", 0.81, "737-800", 13.4, "both channels agree, confidence clears the floor"),
    ("737-600", 0.77, "737-200",  9.1, "the channels disagree"),
    ("737-800", 0.59, "737-800", 13.4, "they agree but the confidence is below the floor"),
    ("737-600", 0.90, "A320",    None, "the airframe is outside the in-service family"),
]
for vv, vc, rv, em, why in demo:
    v, code_, reason = gate_verdict(vv, vc, rv, em)
    print(f"{v:8} {code_:18} {why}")
    print(f"         {reason}\n")

RELEASE  CLEARED            both channels agree, confidence clears the floor
         both channels name 737-800 and confidence 0.81 clears the floor

HOLD     CHANNELS_DISAGREE  the channels disagree
         the stand looks like 737-600, the record says 737-200

HOLD     BELOW_FLOOR        they agree but the confidence is below the floor
         confidence 0.59 is below the floor of 0.60

REFUSE   OUT_OF_FAMILY      the airframe is outside the in-service family
         airframe A320 is outside the in-service family



## 4. The agents

One coordinator holding no tools, three workers holding one capability each. The worker
descriptions are what the coordinator sees when it decides who to ask, and the description
on the visual worker states plainly that its channel is weak by measurement.

That sentence is not there to persuade the model. The constraint is enforced elsewhere, by
the gate. It is there so the reasoning trace records why the channel was to be treated with
caution.

In [4]:
from agents import build_system, decide_one, COORDINATOR_TASK
from smolagents import OpenAIServerModel

model = OpenAIServerModel(model_id=CONFIG["model_id"],
                          api_key=os.environ["OPENAI_API_KEY"])
coordinator, workers = build_system(model)

print("coordinator:", coordinator.name)
print("  tools it holds:", [t for t in coordinator.tools] or "none")
print("  workers it may ask:", list(workers))
for n, w in workers.items():
    print(f"\n  {n}")
    print(f"    holds: {list(w.tools)}")
print("\n\nThe task the coordinator is given, with no facts in it:\n")
print(COORDINATOR_TASK.format(stand="S001"))

coordinator: stand_coordinator
  tools it holds: ['final_answer']
  workers it may ask: ['visual_identification', 'flight_record', 'stand_geometry']

  visual_identification
    holds: ['identify_variant_from_image', 'final_answer']

  flight_record
    holds: ['lookup_expected_variant', 'final_answer']

  stand_geometry
    holds: ['bridge_envelope_for', 'final_answer']


The task the coordinator is given, with no facts in it:

You are the stand coordinator for stand S001. You hold no tools and no facts of your own. Your colleagues hold the facts and you must ask them.
1. Ask visual_identification what the stand camera makes of stand S001, and how confident it is.
2. Ask flight_record which variant the arrival record holds for stand S001.
3. Ask stand_geometry for the door 1L bridge envelope of the variant that FLIGHT RECORD named, not the one the camera named.
Then return your final answer as a single JSON object and nothing else:
{"visual_variant": "...", "visual_confidence": 0.00, 

## 5. A live run, with the agent actions shown

This is the cell that answers whether the system is agentic in fact rather than in
description. It runs three stands live and prints how many steps each agent took and which
tools were called.

If the coordinator were quietly doing the work itself, the worker step count would be
zero.

In [5]:
trace = []
live = [decide_one(r, coordinator, workers, trace) for r in rows[:3]]

for rec in live:
    print(f"{rec.stand}  {rec.verdict:8} {rec.reason_code:18} "
          f"coordinator {rec.coordinator_steps} steps, "
          f"workers {rec.worker_steps} steps, {rec.tool_calls} tool calls")
    print(f"   because {rec.reason}")
    print(f"   coordinator reported faithfully: {rec.reported_faithfully}")
    if rec.fidelity_note:
        print(f"   what it altered: {rec.fidelity_note}")
    print()

S001  HOLD     CHANNELS_DISAGREE  coordinator 4 steps, workers 9 steps, 10 tool calls
   because the stand looks like 737-600, the record says 737-200
   coordinator reported faithfully: False
   what it altered: visual 'Boeing 737-600' not '737-600'; confidence 47 not 0.47

S002  REFUSE   OUT_OF_FAMILY      coordinator 4 steps, workers 9 steps, 10 tool calls
   because airframe A318 is outside the in-service family
   coordinator reported faithfully: False
   what it altered: confidence 44.7 not 0.447

S003  HOLD     CHANNELS_DISAGREE  coordinator 4 steps, workers 9 steps, 10 tool calls
   because the stand looks like 737-600, the record says 737-200
   coordinator reported faithfully: False
   what it altered: visual 'Boeing 737-600' not '737-600'; confidence 75.1 not 0.751



## 6. The full run

The forty stand run was executed separately because it takes several minutes. Its ledger
and evaluation are loaded here rather than re-run, so every number below comes from the
same execution.

To reproduce it: `python3 run.py`

In [6]:
ledger = list(csv.DictReader(open(CONFIG["ledger_file"])))
ev = json.load(open(CONFIG["evaluation_file"]))

print(f"{ev['stands']} stands\n")
for k in ["released", "held", "refused", "release_rate", "hold_rate",
          "precision_when_released", "out_of_family_stands", "out_of_family_released",
          "coordinator_fidelity", "tool_calls_total",
          "mean_coordinator_steps", "mean_worker_steps"]:
    print(f"  {k:28} {ev[k]}")
print("\n  verdicts by reason:")
for k, v in sorted(ev["verdicts_by_reason"].items(), key=lambda x: -x[1]):
    print(f"    {v:3}  {k:28} {REASONS.get(k.split(':')[1], '')}")

40 stands

  released                     7
  held                         24
  refused                      9
  release_rate                 0.175
  hold_rate                    0.6
  precision_when_released      0.8571
  out_of_family_stands         10
  out_of_family_released       0
  coordinator_fidelity         0.025
  tool_calls_total             401
  mean_coordinator_steps       4.0
  mean_worker_steps            9.03

  verdicts by reason:
     22  HOLD:CHANNELS_DISAGREE       the stand and the record name different variants
      9  REFUSE:OUT_OF_FAMILY         the airframe is outside the in-service family
      7  RELEASE:CLEARED              both channels name the same variant and confidence clears the floor
      2  HOLD:BELOW_FLOOR             visual confidence is below the stated floor


## 7. What the coordinator did to the facts in transit

This is the finding, and it is not one I designed for.

The coordinator gathered every fact correctly. It then altered at least one of them on the
way back on thirty nine stands out of forty. The alteration that matters is numeric: a
confidence of 0.47 reported as 47. Against a floor of 0.60, forty seven clears
comfortably and nought point four seven does not.

In [7]:
import re
from collections import Counter

kinds = Counter()
for r in ledger:
    note = r["fidelity_note"] or ""
    if re.search(r"visual '([^']*)' not", note):
        kinds["variant name embellished"] += 1
    if re.search(r"record '([^']*)' not", note):
        kinds["record name embellished"] += 1
    m = re.search(r"confidence ([0-9.eE+-]+) not", note)
    if m:
        src = float(r["visual_confidence"])
        kinds["confidence rescaled to a percentage"
              if abs(float(m.group(1)) - src * 100) < 0.05
              else "confidence value differs"] += 1

drifted = sum(1 for r in ledger if r["fidelity_note"])
print(f"stands where a fact was altered in transit: {drifted} of {len(ledger)}\n")
for k, v in kinds.most_common():
    print(f"  {v:3}  {k}")

print("\nan example, in full:")
ex = next(r for r in ledger if "confidence" in (r["fidelity_note"] or ""))
print(f"  {ex['stand']}")
print(f"    the tools returned : {ex['visual_variant']} at {ex['visual_confidence']}")
print(f"    the coordinator said: {ex['fidelity_note']}")

stands where a fact was altered in transit: 39 of 40

   39  confidence rescaled to a percentage
    4  variant name embellished
    2  record name embellished

an example, in full:
  S001
    the tools returned : 737-600 at 0.47
    the coordinator said: record 'Boeing 737-200' not '737-200'; confidence 47 not 0.47


## 8. The counterfactual

Knowing the coordinator drifts is not the same as knowing whether it mattered. So the same
gate is run a second time over the same ledger, fed the values the coordinator reported
rather than the values the tools returned, and the two sets of verdicts are compared.

This is a measurement over the ledger rather than a component of the system. Putting a
measuring instrument inside the thing it measures is how you end up unable to tell which
one is broken.

In [8]:
def reported_values(r):
    note = r["fidelity_note"] or ""
    vv, rv, vc = r["visual_variant"], r["record_variant"], float(r["visual_confidence"])
    m = re.search(r"visual '([^']*)' not", note)
    if m: vv = m.group(1)
    m = re.search(r"record '([^']*)' not", note)
    if m: rv = m.group(1)
    m = re.search(r"confidence ([0-9.eE+-]+) not", note)
    if m: vc = float(m.group(1))
    em = float(r["envelope_m"]) if r["envelope_m"] not in ("", "None") else None
    return vv, vc, rv, em

differ, unsafe, changes = 0, 0, Counter()
for r in ledger:
    sv, sc, _ = gate_verdict(*reported_values(r))
    if sv != r["verdict"]:
        differ += 1
        changes[f"{r['verdict']}:{r['reason_code']} would have become {sv}:{sc}"] += 1
        if sv == "RELEASE":
            unsafe += 1

print(f"verdicts that would have differed : {differ} of {len(ledger)}")
print(f"differences in the unsafe direction: {unsafe}\n")
for k, v in changes.most_common():
    print(f"  {v:3}  {k}")

verdicts that would have differed : 3 of 40
differences in the unsafe direction: 1

    2  HOLD:CHANNELS_DISAGREE would have become REFUSE:OUT_OF_FAMILY
    1  HOLD:BELOW_FLOOR would have become RELEASE:CLEARED


## 9. Assertions over the ledger

A defect that only shows up when a person looks at a table will eventually be missed. A
defect that fails an assertion will not. These run at the end of every execution.

In [9]:
from evaluation import assertions
from helpers import DecisionRecord

recs = []
for r in ledger:
    rec = DecisionRecord(stand=r["stand"], timestamp=r["timestamp"], truth=r["truth"])
    rec.visual_variant = r["visual_variant"]
    rec.visual_confidence = float(r["visual_confidence"])
    rec.record_variant = r["record_variant"]
    rec.envelope_m = float(r["envelope_m"]) if r["envelope_m"] not in ("", "None") else None
    rec.verdict, rec.reason_code = r["verdict"], r["reason_code"]
    rec.correct = r["correct"] == "True" if r["correct"] not in ("", "None") else None
    rec.schema_rejections = int(r["schema_rejections"])
    rec.coordinator_steps = int(r["coordinator_steps"])
    rec.tool_calls = int(r["tool_calls"])
    rec.reported_faithfully = r["reported_faithfully"] == "True"
    recs.append(rec)

ok = True
for name, passed, detail in assertions(ev, recs):
    print(f"  {'PASS' if passed else 'FAIL'}  {name}  ({detail})")
    ok = ok and passed
print(f"\n  {'all assertions hold' if ok else 'AN ASSERTION FAILED'}")

  PASS  every stand reached a terminal state  (7+24+9 of 40)
  PASS  no out-of-family airframe was released  (0 of 10)
  PASS  no release sits below the confidence floor  (checked per record)
  PASS  no release sits on disagreeing channels  (checked per record)
  PASS  the system does not release everything  (release rate 0.175)
  PASS  the system does release something  (7 released)
  PASS  every verdict carries a reason code  (40 records)
  PASS  the coordinator delegated rather than guessing  (401 calls over 40 stands)
  PASS  a refused tool call never produced a wrong outcome  (0 stands saw a refused call, 0 went wrong)
  PASS  coordinator drift cannot reach the verdict  (the gate reads the tool sources, not the coordinator's copy)

  all assertions hold


## 10. What this run showed

Seven stands released, twenty four held, nine refused. No out-of-family airframe released
on any of the ten it saw. Precision when releasing 0.857, and the one incorrect release is
a case where both channels agreed and both were wrong, which is the failure no amount of
cross-checking between two sources can catch.

The system was built to decide whether automated visual docking guidance may be released
for an arriving aircraft, and to place that decision in a Python gate rather than in a
model, with authority granted to each agent by the tools it holds rather than by anything
its prompt says.

The behaviour that matters was not designed for. The coordinator gathered every fact
correctly and then altered at least one of them on the way back on thirty nine stands out
of forty, most often by rescaling a confidence of 0.47 into 47, which clears a floor of
0.60 comfortably when the original does not.

The hardest part of the build was proving that, because the instrument was wrong before
the thing it measured was: the first fidelity check reported nought faithful out of three
when the true figure was three out of three, since smolagents returns a dictionary and the
parser was reading it as a JSON string. A second difficulty was proving the gate had not
moved across four rewrites of the surrounding code, which is why the ledger's decision
columns are fingerprinted rather than read.

The limitation to state plainly is that the system is not safe because it is well designed.
It is safe because one specific decision was made correctly, to feed the gate from the tool
returns rather than from the coordinator's report of them, and a system whose safety rests
on a single decision has no depth to it. That is the first item in the future improvements
section of the report.
